In [ ]:
from train import data_loaders
from unet import UNet

images_dir = "./dataset"
batch_size = 16
loader_train, loader_valid = data_loaders(images_dir, batch_size)
loaders = {"train": loader_train, "valid": loader_valid}

unet = UNet()

In [8]:
from utils.loss import DiceLoss
from torch import optim

dice_loss = DiceLoss()
loss_train = []
loss_valid = []

optimizer = optim.Adam(unet.parameters(), lr=0.001)

In [ ]:
from utils.visualize import sample_images
import numpy as np

epochs = 10
for epoch in range(epochs):
    for mode in ["train", "valid"]:
        if mode == "train":
            unet.train()
        else:
            unet.eval()
        
        for i, data in enumerate(loaders[mode]):
            optimizer.zero_grad()

            X, y_true = data
            y_pred = unet(X)

            loss = dice_loss(y_true, y_pred)
            if mode == "train":
                loss_train.append(loss.item())
                loss.backward()
                optimizer.step()
            else:
                loss_valid.append(loss.item())
            
            sample_images(
                y_pred.detach().numpy(), 
                y_true.detach().numpy()
            )

    print(f"Epoch {i+1}:  validation loss {np.mean(loss_valid)}")
    loss_valid = []